#   Arabic Tashkeel (تشكيل) with Seq2Seq Transformer

In [1]:
from typing import List
def compute_length_stats(sources: List[str], targets: List[str]) -> dict:
    """
    WHY: Before setting MAX_SRC_LEN and MAX_TGT_LEN, run this on your
    actual dataset. Don't guess — measure.
 
    Print the 95th percentile of source and target lengths, then set
    your limits to the 99th percentile (to cover 99% of samples without
    truncation). The remaining 1% will be truncated — that's acceptable.
    """
    import numpy as np
 
    src_lens = [len(s) for s in sources]
    tgt_lens = [len(t) for t in targets]
    ratios   = [len(t) / max(len(s), 1) for s, t in zip(sources, targets)]
 
    stats = {
        "src_p50":  int(np.percentile(src_lens, 50)),
        "src_p95":  int(np.percentile(src_lens, 95)),
        "src_p99":  int(np.percentile(src_lens, 99)),
        "tgt_p50":  int(np.percentile(tgt_lens, 50)),
        "tgt_p95":  int(np.percentile(tgt_lens, 95)),
        "tgt_p99":  int(np.percentile(tgt_lens, 99)),
        "ratio_mean": float(round(sum(ratios) / len(ratios), 2)),
        "ratio_max":  float(round(max(ratios), 2)),
    }
 
    print("=== Dataset Length Statistics ===")
    print(f"  Source p50={stats['src_p50']}  p95={stats['src_p95']}  p99={stats['src_p99']}")
    print(f"  Target p50={stats['tgt_p50']}  p95={stats['tgt_p95']}  p99={stats['tgt_p99']}")
    print(f"  tgt/src ratio: mean={stats['ratio_mean']}  max={stats['ratio_max']}")
    print(f"\n  → Recommended MAX_SRC_LEN = {stats['src_p99'] + 20}")
    print(f"  → Recommended MAX_TGT_LEN = {stats['tgt_p99'] + 40}")
 
    return stats


## Tokenizer


In [2]:
#   Special tokens:
#       <PAD> = 0   — padding to batch variable-length sequences
#       <SOS> = 1   — start-of-sequence (decoder prompt)
#       <EOS> = 2   — end-of-sequence (decoder stop signal)
#       <UNK> = 3   — unknown character (fallback)
from typing import Dict, List
class ArabicCharTokenizer:
    PAD, SOS, EOS, UNK = 0, 1, 2, 3
    SPECIAL = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
    
    def __init__(self):
        self.char2id: Dict[str,int]={}
        self.id2char: Dict[int,str]={}
        for i ,tok in enumerate(self.SPECIAL):
            self.char2id[tok]= i
            self.id2char[i]=tok

    def build_vocab(self, texts:List[str])->None:
        chars= set()
        for text in texts:
            chars.update(text)
        chars= sorted(chars) # deterministic order

        start_id= len(self.SPECIAL)
        for i, ch in enumerate(chars):
            self.char2id[ch] = start_id +i
            self.id2char[start_id+i]= ch
        print(f"[Tokenizer] Vocabulary size: {len(self.char2id)}") 
        
    
    def encode(self, text:str,add_sos:bool=False, add_eos:bool=True)-> List[int]:
        ids= []
        if add_sos:
            ids.append(self.SOS)
        for ch in text:
            ids.append(self.char2id.get(ch,self.UNK))
        if add_eos:
            ids.append(self.EOS)
        return ids
    def decode(self, ids:List[int], skip_speicals:bool=True)->str:
        chars=[]
        for i in ids:
            if skip_speicals and i in (self.PAD,self.SOS,self.EOS,self.UNK):
                continue
            chars.append(self.id2char.get(i,"?"))
        return "".join(chars)
   
    @property
    def vocab_size(self) -> int:
        return len(self.char2id)
    @property
    def vocabs(self)->Dict[str,int]:
        return self.char2id

## Dataset

In [3]:
from torch.utils.data import Dataset, DataLoader
import torch
from typing import List
#from .tokenizer import ArabicCharTokenizer
MAX_SRC_LEN=150 #max undiacritized chars per sample
MAX_TGT_LEN=500 #  max diacritized chars (includes harakat between chars)


class TashkeelDataset(Dataset):
    """Handle batching, shuffling and multiworker.

    Return:
        src_ids — encoded undiacritized source, padded to MAX_SRC_LEN
        tgt_ids — encoded diacritized target (WITH <SOS> prepended), padded
        tgt_labels — same as tgt_ids but SHIFTED LEFT by 1 (for teacher forcing)
    """
    def __init__(self,sources:List[str],
                 targets:List[str],
                 tokenizer:ArabicCharTokenizer,
                 max_src_len:int=MAX_SRC_LEN,
                 max_tgt_len:int=MAX_TGT_LEN):
        super().__init__()
        self.tokenizer=tokenizer
        self.max_src_len=max_src_len
        self.max_tgt_len=max_tgt_len

        self.pairs=[]
        for src,tgt in zip(sources,targets):
            src_ids=tokenizer.encode(src,add_sos=False,add_eos=True)
            # tgt_ids= tokenizer.encode(tgt,add_sos=True,add_eos=True)
            # #Labels = target shifted left 
            # tgt_labels=tokenizer.encode(tgt,add_sos=False,add_eos=True)
            tgt_full = tokenizer.encode(tgt, add_sos=False, add_eos=True)
            tgt_ids = [tokenizer.SOS] + tgt_full[:-1]
            # Labels = target shifted left (no SOS, but EOS included)
            tgt_labels = tgt_full
            #Truncate
            src_ids=src_ids[:max_src_len]
            tgt_ids=tgt_ids[:max_tgt_len]
            tgt_labels = tgt_labels[:max_tgt_len]

            self.pairs.append(
                (src_ids,tgt_ids,tgt_labels)
            )
    

    def __len__(self)->int:
        return len(self.pairs)
    
    def __getitem__(self,idx:int):

        src_ids,tgt_ids,tgt_lables= self.pairs[idx]
        return{
            "src_ids":torch.tensor(src_ids,dtype=torch.long),
            "tgt_ids":torch.tensor(tgt_ids,dtype=torch.long),
            "tgt_labels":torch.tensor(tgt_lables,dtype=torch.long)
        }
    



def collate_fn(batch:List[dict])->dict:
    """pad shorter ones,
    to the length of the longest in the batch (dynamic padding)."""

    PAD= ArabicCharTokenizer.PAD

    src_padded=torch.nn.utils.rnn.pad_sequence(
        [item['src_ids'] for item in batch],
        batch_first=True,padding_value=PAD
    )

    tgt_padded= torch.nn.utils.rnn.pad_sequence(
        [item['tgt_ids'] for item in batch],
        batch_first=True, padding_value=PAD
    )

    lbl_padded= torch.nn.utils.rnn.pad_sequence(
        [item['tgt_labels'] for item in batch],
        batch_first=True,padding_value=PAD
    )

    #padding mask
    src_key_padding_mask= (src_padded==PAD) 
    tgt_key_padding_mask= (tgt_padded==PAD)

    return{
        "src":src_padded,
        "tgt":tgt_padded,
        "labels":lbl_padded,
        "src_key_padding_mask":src_key_padding_mask,
        "tgt_key_padding_mask":tgt_key_padding_mask
        }

## scheduler

In [4]:
import torch
class WarmupCosineScheduler:
    """Custom LR Scehdular
    """

    def __init__(self,
                 optimizer:torch.optim.Optimizer,
                 d_model:int,
                 warmup_steps:int=400):
        self.optimizer= optimizer
        self.d_model=d_model
        self.warmup_steps= warmup_steps
        self._step=0

    def step(self):
        self._step +=1
        lr= self._compute_lr()
        for param_group in self.optimizer.param_groups:
            param_group['lr']= lr

    def _compute_lr(self)->float:
        step= max(self._step,1)
        return(
            self.d_model ** (-0.5)
            * min(step ** (-0.5), step *  self.warmup_steps ** (-1.5) )
        )
        
        

## Model

### Positional Encoding

In [5]:

import torch
import math
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model:int,max_len:int=500,dropout:float=0.1):
        super().__init__()
        self.dropout=torch.nn.Dropout(dropout)

        pe= torch.zeros(max_len,d_model)
        position= torch.arange(0,max_len,dtype=torch.float).unsqueeze(1) #(max_len, 1)

        div_term=torch.exp(
            torch.arange(0,d_model,2,dtype=torch.float)
            * (-math.log(10000.0)/ d_model)
        )
        #use sin for even dims & consin for odd
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe= pe.unsqueeze(0)
        self.register_buffer("pe",pe) #as a part of state

    def forward(self,x:torch.Tensor)->torch.Tensor:
       x = x + self.pe[:, :x.size(1), :]
       return self.dropout(x)


### Transformer

In [6]:
import torch
import math
from typing import Optional

class TashkeelTransformer(torch.nn.Module):
    def __init__(self,
                src_vocab_size:int,
                tgt_vocab_size:int,
                d_model:int=256,
                n_heads:int=8,
                num_encoder_layers:int=3,
                num_decoder_layers:int=3,
                dim_feedforward:int=512,
                dropout:float=0.1,
                max_seq_len:int=500,
                pad_idx:int=0
                ):
        super().__init__()
        self.d_model= d_model
        self.pad_idx= pad_idx
        self.tgt_vocab_size= tgt_vocab_size

        # Embedding

        self.src_embedding= torch.nn.Embedding(src_vocab_size,d_model,padding_idx=pad_idx)
        self.tgt_embedding= torch.nn.Embedding(tgt_vocab_size,d_model,padding_idx=pad_idx)
        self.scale= math.sqrt(d_model) 
        
        self.pos_encoding= PositionalEncoding(d_model,max_seq_len,dropout)


        # Transformer

        self.transformer= torch.nn.Transformer(
            d_model=d_model,
            nhead=n_heads,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.output_projection= torch.nn.Linear(d_model,tgt_vocab_size)
        self.output_projection.weight = self.tgt_embedding.weight
        self._init_weights()


    def _init_weights(self):
        for p in self.parameters():
            if p.dim() >1 :
                torch.nn.init.xavier_uniform_(p)
        torch.nn.init.normal_(self.src_embedding.weight,mean=0, std=0.01)
        torch.nn.init.normal_(self.tgt_embedding.weight,mean=0,std=0.01)

    @staticmethod
    def make_causal_mask(sz:int,device:torch.device) -> torch.Tensor:
        mask= torch.triu(torch.ones(sz,sz,device=device),diagonal=1)
        return mask.masked_fill(mask==1 ,float("-inf"))

    def encode(self,
               src:torch.Tensor,
               src_key_padding_mask:Optional[torch.Tensor]=None)->torch.Tensor:
        src_emb= self.pos_encoding(self.src_embedding(src) * self.scale)
        return self.transformer.encoder(
            src_emb,
            src_key_padding_mask=src_key_padding_mask)
    

    def decoder(self,
                tgt:torch.Tensor,
                memory:torch.Tensor,
                tgt_mask:Optional[torch.Tensor]=None,
                tgt_key_padding_mask:Optional[torch.Tensor]=None,
                memory_key_padding_mask:Optional[torch.Tensor]=None)->torch.Tensor:
        tgt_emb= self.pos_encoding(self.tgt_embedding(tgt) * self.scale)

        return self.transformer.decoder(
            tgt_emb,
            memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )   
    

    def forward(self,
                src:torch.Tensor,
                tgt:torch.Tensor,
                src_key_padding_mask: Optional[torch.Tensor] = None,
                tgt_key_padding_mask: Optional[torch.Tensor] = None,
                )-> torch.Tensor:
        tgt_len = tgt.size(1)
        device = src.device

        tgt_mask = self.make_causal_mask(tgt_len, device)
        memory = self.encode(src, src_key_padding_mask)

        decoder_out = self.decoder(
            tgt, memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )

        logits = self.output_projection(decoder_out) 
        return logits


## metrics

In [7]:
import editdistance
from typing import List, Tuple, Dict
ARABIC_LETTERS = set(chr(c) for c in range(0x0621, 0x064B))
HARAKAT = {
    "\u064E",   
    "\u064F",   
    "\u0650",   
    "\u0651",  
    "\u0652",   
    "\u064B",   
    "\u064C",   
    "\u064D",   
    "\u0670",   
}

def extract_char_diacritic_pairs(text:str)-> List[Tuple[str,str]]:
    """Extract (base_char, diacritic) pairs from diacritized Arabic text.

    Returns:
        List of (base_char, diacritic_or_empty_string) tuples.
    """
    pairs= []
    i=0
    while i < len(text):
        ch= text[i]
        if ch in ARABIC_LETTERS:
            diac = ""

            j= i+1
            while j < len(text) and text[j] in HARAKAT:
                diac += text[j]
                j+= 1

            pairs.append((ch,diac))
            i=j
        else:
            i +=1
    return  pairs



def compute_der(predicted:str,reference:str)->float:
    pred_pairs= extract_char_diacritic_pairs(predicted)
    ref_pairs= extract_char_diacritic_pairs(reference)

    n= min(len(pred_pairs), len(ref_pairs))

    if n== 0:
        return 1.0
    errors= sum(
        1 for (pc,pd ), (rc,rd) in zip(pred_pairs[:n], ref_pairs[:n])
        if pd !=rd # diacritic mismetch 
    )
    
    return errors / n 

def evaluate_corpus_der(predictions:List[str],
                        reference:List[str])-> Dict[str,float]:

                        ders= [compute_der(p,r) for p,r in zip(predictions,reference)]

                        return {"DER_mean": sum(ders) / len(ders),
                                "DER_min": min(ders),
                                "DER_max": max(ders),
                                "n_samples": len(ders),
                        }

def exact_match(predictions, references):
    correct = sum(p == r for p, r in zip(predictions, references))
    return correct / len(predictions)


def cer(predicted: str, reference: str) -> float:
    return editdistance.eval(predicted, reference) / len(reference)

## greedy search decode

In [8]:
import torch
import torch.nn.functional as F
from typing import List

MAX_TGT_LEN=500
@torch.no_grad()
def greedy_decode(model:TashkeelTransformer,
                       src_ids:List[int],
                       tokenizer:ArabicCharTokenizer,
                       device=torch.device,
                       beam_search:int=4,
                       max_len:int=MAX_TGT_LEN)->str: 

    model.eval()

    src=torch.tensor([src_ids], dtype=torch.long, device=device) 
    memory= model.encode(src)
    

    generated= [tokenizer.SOS]
    
    for _ in range(max_len):
        tgt= torch.tensor([generated], dtype=torch.long,device=device)
        tgt_len= tgt.size(1)

        tgt_mask= model.make_causal_mask(tgt_len,device)

        decoder_out= model.decoder(tgt,memory,tgt_mask=tgt_mask)
        logits= model.output_projection(decoder_out[:,-1, :])

        next_token= logits.argmax(dim=-1).item()
        generated.append(next_token)

        if next_token== tokenizer.EOS: break
    
    return tokenizer.decode(generated)


## Beam search decode


In [9]:
    @torch.no_grad()
    def beam_search_decode(
        model: TashkeelTransformer,
        src_ids: List[int],
        tokenizer: ArabicCharTokenizer,
        device: torch.device,
        beam_size: int = 4,
        max_len: int = MAX_TGT_LEN,
        length_penalty: float = 0.6,
    ) -> str:
        """
        Beam search decoder.
     
        LENGTH PENALTY: log P / (len^alpha)
        Without this, beam search prefers SHORT sequences (summing log probs
        penalizes long sequences). alpha=0.6 is a common setting.
        """
        model.eval()
     
        src = torch.tensor([src_ids], dtype=torch.long, device=device)
        memory = model.encode(src)  # (1, src_len, d_model)
     
        # Each beam: (sequence of token IDs, cumulative log probability)
        beams: List[Tuple[List[int], float]] = [([tokenizer.SOS], 0.0)]
        completed: List[Tuple[List[int], float]] = []
     
        for step in range(max_len):
            if not beams:
                break
     
            candidates: List[Tuple[List[int], float]] = []
     
            for seq, score in beams:
                tgt = torch.tensor([seq], dtype=torch.long, device=device)
                tgt_mask = model.make_causal_mask(len(seq), device)
     
                decoder_out = model.decoder(tgt, memory, tgt_mask=tgt_mask)
                logits = model.output_projection(decoder_out[:, -1, :])  # (1, V)
                log_probs = F.log_softmax(logits, dim=-1).squeeze(0)     # (V,)
     
                # Expand top beam_size candidates from this beam
                topk_log_probs, topk_ids = log_probs.topk(beam_size)
                for log_p, tok_id in zip(topk_log_probs.tolist(), topk_ids.tolist()):
                    new_seq = seq + [tok_id]
                    new_score = score + log_p
     
                    if tok_id == tokenizer.EOS:
                        # Penalize by length to prevent short-sequence bias
                        normalized = new_score / (len(new_seq) ** length_penalty)
                        completed.append((new_seq, normalized))
                    else:
                        candidates.append((new_seq, new_score))
     
            # Keep only top beam_size active beams
            candidates.sort(key=lambda x: x[1], reverse=True)
            beams = candidates[:beam_size]
     
        # If no complete beams, take the best active beam
        if not completed:
            completed = [(seq, score / (len(seq) ** length_penalty)) for seq, score in beams]
     
        # Return the best complete beam
        best_seq = max(completed, key=lambda x: x[1])[0]
        return tokenizer.decode(best_seq)


## Train loop


In [10]:

from torch.utils.data import DataLoader
import torch
import os
import sys
from tqdm.auto import tqdm

def train_epoch(model: TashkeelTransformer,
                dataloader:DataLoader,
                optimizer: torch.optim.Optimizer,
                scheduler,
                criterion:torch.nn.CrossEntropyLoss,
                 device:torch.device,
                  clip_grad_norm:float = 1.0 )->float:
    
    model.train()
    total_loss=0.0
    n_batches=0

    for batch in dataloader:
        src= batch['src'].to(device)
        tgt= batch['tgt'].to(device)
        labels= batch['labels'].to(device)
        src_mask = batch["src_key_padding_mask"].to(device)
        tgt_mask = batch["tgt_key_padding_mask"].to(device)

        logits= model(src,tgt,src_key_padding_mask=src_mask)

        B, T,V= logits.shape

        loss= criterion(logits.reshape(B * T, V) ,labels.reshape(B*T))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),clip_grad_norm)
        optimizer.step()

        if scheduler is not None:
            scheduler.step()
        
        total_loss += loss.item()
        n_batches +=1
    return total_loss / max(n_batches, 1)


def evaluate(model: TashkeelTransformer,
             dataloader: DataLoader,
             criterion: torch.nn.CrossEntropyLoss,
             device:torch.device)-> float:
    model.eval()
    total_loss= 0.0
    n_batches=0

    for batch in dataloader:
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)
        labels = batch["labels"].to(device)
        src_mask = batch["src_key_padding_mask"].to(device)
 
        logits = model(src, tgt, src_key_padding_mask=src_mask)
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B * T, V), labels.reshape(B * T))
 
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

def train(model:TashkeelTransformer,
          train_loader:DataLoader,
          val_loader:DataLoader,
          device: torch.device,
          num_epochs:int =20,
          learning_rate:float= 1e-4,
          checkpoint_dir:str="checkpoints"):
    os.makedirs(checkpoint_dir,exist_ok=True)

    optimizer= torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        betas=(0.9, 0.98),   #Transformer paper values
        eps=1e-9,
        weight_decay=0.01
    )

    scheduler= WarmupCosineScheduler(optimizer=optimizer,d_model=model.d_model)
    criterion = torch.nn.CrossEntropyLoss(
        ignore_index=ArabicCharTokenizer.PAD,
        label_smoothing=0.1,
    )

    best_val_loss = float("inf")

    for epoch in tqdm(range(1, num_epochs +1)):
        train_loss= train_epoch(model,train_loader,optimizer,scheduler,criterion,device)
        val_loss= evaluate(model,val_loader,criterion,device)

        print(
            f"Epoch {epoch:3d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

        if val_loss < best_val_loss:
            best_val_loss= val_loss
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": val_loss,
                },
                os.path.join(checkpoint_dir, "best_model.pt"),
            )
            print(f"Saved best model (val_loss={val_loss:.4f})")




In [11]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
D_MODEL = 256
NUM_EPOCHS = 30
TRAIN_SPLIT = 0.8

## Get data

import json
with open("/kaggle/input/datasets/iamagamy/arabic-sentences-diacritics/data.json", 'r')as f:
    data= json.load(f)


sources = [s for item in data for s in item["src"]]

targets = [t for item in data for t in item["tgt"]]

compute_length_stats(sources,targets)

=== Dataset Length Statistics ===
  Source p50=90  p95=124  p99=173
  Target p50=147  p95=196  p99=199
  tgt/src ratio: mean=1.6  max=2.33

  → Recommended MAX_SRC_LEN = 193
  → Recommended MAX_TGT_LEN = 239


{'src_p50': 90,
 'src_p95': 124,
 'src_p99': 173,
 'tgt_p50': 147,
 'tgt_p95': 196,
 'tgt_p99': 199,
 'ratio_mean': 1.6,
 'ratio_max': 2.33}

In [12]:
len(sources[0])

117

In [13]:
len(targets[0])

193

## Train

In [14]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
D_MODEL = 256
NUM_EPOCHS = 60
TRAIN_SPLIT = 0.8

## Get data

import json
with open("/kaggle/input/datasets/iamagamy/arabic-sentences-diacritics/data.json", 'r')as f:
    data= json.load(f)
sources = [s for item in data for s in item["src"]]

targets = [t for item in data for t in item["tgt"]]

print(f"Loaded {len(sources)} sentence pairs.")
print(f"Example:\n  SRC: {sources[0]}\n  TGT: {targets[0]}")

tokenizer = ArabicCharTokenizer()
tokenizer.build_vocab(sources + targets)  # build from BOTH sides
print(f"Vocab size: {tokenizer.vocab_size}")


n = len(sources)
split = int(n * TRAIN_SPLIT) #Type Markdown and LaTeX: 
train_src, train_tgt = sources[:split], targets[:split]
val_src, val_tgt = sources[split:], targets[split:]

train_dataset = TashkeelDataset(train_src, train_tgt, tokenizer)
val_dataset = TashkeelDataset(val_src, val_tgt, tokenizer)

train_loader = DataLoader(
  train_dataset, batch_size=BATCH_SIZE, shuffle=True,
  collate_fn=collate_fn, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
  val_dataset, batch_size=BATCH_SIZE, shuffle=False,
  collate_fn=collate_fn, num_workers=2, pin_memory=True
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## build model
model = TashkeelTransformer(
        src_vocab_size=tokenizer.vocab_size,
        tgt_vocab_size=tokenizer.vocab_size,
        d_model=D_MODEL,
        n_heads=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=512,
        dropout=0.1,
        pad_idx=ArabicCharTokenizer.PAD,
    ).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")


train(model, train_loader, val_loader, DEVICE, num_epochs=NUM_EPOCHS)

Loaded 52570 sentence pairs.
Example:
  SRC: جميع الحقوق محفوظة قناة أولو العلم . يسرنا إتاحة هذا الكتاب للنشر والمشاركة بنية نشر العلم الشرعي والمعرفة الإسلامية،
  TGT: جَمِيعُ الْحُقُوقِ مَحْفُوظَةٌ قَنَاةِ أُولُو الْعِلْمِ . يَسُرُّنَا إِتَاحَةُ هَذَا الكِتَابِ لِلنَّشْرِ وَالمُشَارَكَةِ بِنِيَّةِ نَشْرِ العِلْمِ الشَّرْعِيِّ وَالمَعْرِفَةِ الإِسْلَامِيَّةِ،
[Tokenizer] Vocabulary size: 56
Vocab size: 56
Train batches: 1315 | Val batches: 329


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


Model parameters: 3,983,416


  0%|          | 0/60 [00:00<?, ?it/s]

Epoch   1 | Train Loss: 1.8430 | Val Loss: 0.9773 | LR: 1.72e-03
Saved best model (val_loss=0.9773)
Epoch   2 | Train Loss: 0.9752 | Val Loss: 0.8747 | LR: 1.22e-03
Saved best model (val_loss=0.8747)
Epoch   3 | Train Loss: 0.8994 | Val Loss: 0.8469 | LR: 9.95e-04
Saved best model (val_loss=0.8469)
Epoch   4 | Train Loss: 0.8698 | Val Loss: 0.8311 | LR: 8.62e-04
Saved best model (val_loss=0.8311)
Epoch   5 | Train Loss: 0.8528 | Val Loss: 0.8228 | LR: 7.71e-04
Saved best model (val_loss=0.8228)
Epoch   6 | Train Loss: 0.8415 | Val Loss: 0.8151 | LR: 7.04e-04
Saved best model (val_loss=0.8151)
Epoch   7 | Train Loss: 0.8331 | Val Loss: 0.8107 | LR: 6.51e-04
Saved best model (val_loss=0.8107)
Epoch   8 | Train Loss: 0.8264 | Val Loss: 0.8064 | LR: 6.09e-04
Saved best model (val_loss=0.8064)
Epoch   9 | Train Loss: 0.8211 | Val Loss: 0.8034 | LR: 5.75e-04
Saved best model (val_loss=0.8034)
Epoch  10 | Train Loss: 0.8168 | Val Loss: 0.8009 | LR: 5.45e-04
Saved best model (val_loss=0.8009)


## Eval

In [15]:
from tqdm.auto import tqdm
def evaluate_model(model, sources, targets, tokenizer, device, max_samples=None):
    predictions = []
    references = []

    if max_samples:
        sources = sources[:max_samples]
        targets = targets[:max_samples]

    for src_text, tgt_text in tqdm(zip(sources, targets)):
        src_ids = tokenizer.encode(src_text)

        pred = beam_search_decode(model, src_ids, tokenizer, device)

        predictions.append(pred)
        references.append(tgt_text)

    return {
        "DER": evaluate_corpus_der(predictions, references),
       "CER": sum(cer(p, r) for p, r in zip(predictions, references)) / len(predictions),
        "ExactMatch": exact_match(predictions, references),
    }
device= 'cuda' 
eval_result= evaluate_model(model=model,
                           sources=sources,
                           targets=targets,
                           tokenizer=tokenizer,
                           device=device,
                           max_samples=100)

0it [00:00, ?it/s]

In [16]:
eval_result

{'DER': {'DER_mean': 0.03579124613718253,
  'DER_min': 0.0,
  'DER_max': 0.21052631578947367,
  'n_samples': 100},
 'CER': 0.02033604784724756,
 'ExactMatch': 0.4}

In [17]:
import json
from pathlib import Path

Path("assets").mkdir(exist_ok=True)


tokenizer_data = {
    "char2id": tokenizer.char2id,
    "id2char": {str(k): v for k, v in tokenizer.id2char.items()},
    "vocab_size": tokenizer.vocab_size
}
with open("assets/tokenizer.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer_data, f, ensure_ascii=False, indent=2)


config = {
    "d_model": 256, 
    "nhead": 8, 
    "num_encoder_layers": 3,
    "num_decoder_layers": 3, 
    "dim_feedforward": 512, 
    "dropout": 0.1,
    "max_src_len": 150, 
    "max_tgt_len": 500,
    "vocab_size": tokenizer.vocab_size  # ← THIS WAS MISSING
}
with open("assets/config.json", "w") as f:
    json.dump(config, f, indent=2)

In [18]:
import re
class ArabicTextChunker:
    SENTENCE_ENDINGS = re.compile(r'([.؟!،]\s*)')

    def __init__(self,
                 max_chunk_size:int=140,
                 min_chunk_size:int=30,         
                overlap_chars:int=30
                 ):
    
        self.max_chunk_size=max_chunk_size
        self.min_chunk_size=min_chunk_size
        self.overlap_chars=overlap_chars
    

    def chunk(self, text:str)->List[str]:
        if len(text) <=self.max_chunk_size:
            return [text.strip()]
    

        chunks= []
        start=0
        while start < len(text):
            end= start + self.max_chunk_size

            if end>= len(text):
                ch= text[start:].strip()
                if ch:
                    chunks.append(ch)
                break


            segment= text[start:end]
            split_point= self._find_best_split(segment,start)

            if split_point is None:
                last_space= segment.rfind(' ')
                if last_space > self.min_chunk_size:
                    split_point= start + last_space +1
                else:
                    split_point = end


             
            

            ch = text[start:split_point].strip()
            if ch:
                chunks.append(ch)
            
            if self.overlap_chars > 0 and start > 0:
                start= split_point - self.overlap_chars
                if start < 0:
                    start= 0 
            else:
                start = split_point

        return chunks

    
    def _find_best_split(self,segment:str,global_start:int)->int:
        for i in range(len(segment) - 1, self.min_chunk_size - 1, -1):
            if segment[i] in '.؟!،':
                return global_start + i + 1  
        return None
    
  

In [19]:
class TashkeelPredictor:
    def __init__(self,
                  checkpoint_path="checkpoints",
                  assets_path="assets",
                    device=None):
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        
        # Load config & tokenizer
        with open(f"{assets_path}/config.json", "r") as f:
            cfg = json.load(f)
        with open(f"{assets_path}/tokenizer.json", "r", encoding="utf-8") as f:
            tok_data = json.load(f)
        
        ## Define the txt chunker
        self.chunker = ArabicTextChunker(
           
          overlap_chars=0
        )
        
        self.tokenizer = ArabicCharTokenizer()
        self.tokenizer.char2id = {k: int(v) for k, v in tok_data["char2id"].items()}
        self.tokenizer.id2char = {int(k): v for k, v in tok_data["id2char"].items()}
        self.max_src_len = cfg["max_src_len"]
        
        # Initialize model with saved hyperparameters
        self.model = TashkeelTransformer(
            src_vocab_size=cfg["vocab_size"], tgt_vocab_size=cfg["vocab_size"],
            d_model=cfg["d_model"], n_heads=8,
            num_encoder_layers=cfg["num_encoder_layers"],
            num_decoder_layers=cfg["num_decoder_layers"],
            dim_feedforward=cfg["dim_feedforward"],
            dropout=cfg["dropout"], pad_idx=ArabicCharTokenizer.PAD
        ).to(self.device)
        
        # Load weights
        ckpt = torch.load(f"{checkpoint_path}/best_model.pt", map_location=self.device, weights_only=True)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.eval()
        print(f"Model loaded on {self.device}")


    def _diacritize_single(self, text: str, beam_size: int = 4)->str:
        src_ids = self.tokenizer.encode(text, add_sos=False, add_eos=True)
        max_len = self.max_src_len
        if len(src_ids) > max_len:
            src_ids = src_ids[:max_len]
        
        with torch.no_grad():
            return beam_search_decode(
                self.model, src_ids, self.tokenizer, self.device, beam_size=beam_size
            )
           


    def diacritize(self, text: str, use_beam: bool = True, beam_size: int = 4) -> str:
        """Diacritize a single Arabic sentence."""
        if len(text.strip()) == 0: return ""
        
        if len(text.strip()) <= self.chunker.max_chunk_size:
            return self._diacritize_single(text.strip(), beam_size)

        chunks= self.chunker.chunk(text.strip())
        print(f"chunks :{chunks}")
        diacritized_chunks = [
            self._diacritize_single(chunk,  beam_size)
            for chunk in chunks
        ]
        return " ".join(diacritized_chunks)


    def diacritize_batch(self, texts: list, use_beam: bool = True, beam_size: int = 4) -> list:
        """Process multiple sentences efficiently."""
        return [self.diacritize(t, use_beam, beam_size) for t in texts]
    

In [20]:
predictor = TashkeelPredictor()

text = """
هذا من فضل ربي
""".strip()

result = predictor.diacritize(text, use_beam=True, beam_size=4)
print(f"Input:  {text}")
print(f"Output: {result}")


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


Model loaded on cuda
Input:  هذا من فضل ربي
Output: هَذَا مِنْ فَضْلِ رَبِّي


In [ ]:
!zip -r model.zip  "/kaggle/working/assets" "/kaggle/working/checkpoints"